# 🔧 Patch Cells v3 — Uses YOUR Exact Variable Names

**All variable names kept identical:** `FEATURES`, `X_encoded`, `X_const`, `y_ord`, `y_count`, `y_bin`, `ord_result`, `poisson_m`, `nb_m`, `best_count_m`, `best_count_name`, `logit_m`, `chi2_df`, `binary_or`, etc.

**How to use:** Run your original notebook through **cell 2.9** (cleaning step). Then paste these patches in order.


## PATCH 1: Collapse Parity → 1, 2, 3, 4+
📌 After cell 2.9


In [ ]:
# PATCH 1: Collapse parity
df['parity_collapsed'] = df['parity'].apply(lambda x: int(min(x, 4)) if pd.notna(x) else np.nan)

print('COLLAPSED PARITY:')
print(df['parity_collapsed'].value_counts().sort_index())
print('1=First, 2=Second, 3=Third, 4=Fourth+')


## PATCH 2: Binary Multiple Birth
📌 After Patch 1


In [ ]:
# PATCH 2: Collapse multiple_birth → binary
# Keeps 'multiple_birth' column untouched, adds new column
df['is_multiple'] = (df['multiple_birth'].astype(str).str.strip().str.lower() != 'singleton').astype(int)

print('is_multiple:')
print(df['is_multiple'].value_counts().rename({0:'Singleton', 1:'Multiple'}))


## PATCH 3: Add Province
📌 After Patch 2


In [ ]:
# PATCH 3: Province from district_mother
DIST_PROV = {
    'Colombo':'Western','Gampaha':'Western','Kalutara':'Western',
    'Kandy':'Central','Matale':'Central','Nuwara Eliya':'Central','Nuwara-Eliya':'Central',
    'Galle':'Southern','Matara':'Southern','Hambantota':'Southern',
    'Jaffna':'Northern','Kilinochchi':'Northern','Mannar':'Northern',
    'Mullaitivu':'Northern','Vavuniya':'Northern',
    'Batticaloa':'Eastern','Ampara':'Eastern','Trincomalee':'Eastern',
    'Kurunegala':'North Western','Puttalam':'North Western',
    'Anuradhapura':'North Central','Polonnaruwa':'North Central',
    'Badulla':'Uva','Monaragala':'Uva','Moneragala':'Uva',
    'Ratnapura':'Sabaragamuwa','Kegalle':'Sabaragamuwa',
}

df['district_clean'] = df['district_mother'].astype(str).str.strip().str.title()
df['province'] = df['district_clean'].map(DIST_PROV)

print('Province:')
print(df['province'].value_counts().sort_index())
unmapped = [d for d in df[df['province'].isna()]['district_clean'].unique() if d != 'Nan']
if unmapped:
    print(f'⚠ Unmapped: {unmapped}')

before = len(df)
df = df.dropna(subset=['province']).reset_index(drop=True)
print(f'Rows: {before:,} → {len(df):,}')


## PATCH 4: GVIF (Replaces VIF cell)
📌 Replace cell 4.3 (the VIF cell)

Uses the same `FEATURES` list + new variables. Does NOT rename anything.


In [ ]:
# PATCH 4: GVIF — replaces VIF (cell 4.3)
# Same FEATURES list as your original, plus new ones

import numpy as np
import pandas as pd

# Your original FEATURES list (unchanged)
FEATURES_ORIGINAL = ['age_group', 'marital_status', 'race_mother',
                     'Gender ', 'Hospital or Not', 'multiple_birth']

# New variables added by supervisor's comments
NEW_VARS = ['province', 'is_multiple']

# Separate categorical vs continuous
# ALL of FEATURES_ORIGINAL are categorical (object dtype in df)
# is_multiple is binary (int) but treat as categorical for GVIF
# Birth_Weight(grams) is the only continuous variable

CAT_VARS = FEATURES_ORIGINAL + ['province']  # all categorical
CONT_VARS = ['Birth_Weight(grams)']
BIN_VARS = ['is_multiple']  # binary, treat as categorical

# Check they exist
print("Checking columns exist:")
for v in CAT_VARS + CONT_VARS + BIN_VARS:
    exists = v in df.columns
    print(f"  {'✓' if exists else '✗'} {v}")

# Build design matrix for GVIF
parts = []

# Categorical → dummies (drop_first)
parts.append(pd.get_dummies(df[CAT_VARS], drop_first=True, dtype=float))

# Binary → convert to string then dummy
for b in BIN_VARS:
    b_str = df[b].astype(str)
    parts.append(pd.get_dummies(b_str, prefix=b, drop_first=True, dtype=float))

# Continuous → as-is
for c in CONT_VARS:
    parts.append(df[[c]].astype(float))

X_gvif = pd.concat(parts, axis=1).dropna()
col_list = list(X_gvif.columns)
print(f"\nDesign matrix for GVIF: {X_gvif.shape}")

# Map each variable → its dummy columns
var_to_cols = {}
for v in CAT_VARS:
    var_to_cols[v] = [c for c in col_list if c.startswith(v + '_')]
for v in BIN_VARS:
    var_to_cols[v] = [c for c in col_list if c.startswith(v + '_')]
for v in CONT_VARS:
    var_to_cols[v] = [v]

# Compute GVIF
R = X_gvif.corr().values
cond = np.linalg.cond(R)
R_inv = np.linalg.pinv(R) if cond > 1e10 else np.linalg.inv(R)

gvif_rows = []
for var, cols in var_to_cols.items():
    d = len(cols)
    idx = [col_list.index(c) for c in cols]
    
    gvif = abs(np.linalg.det(R_inv[np.ix_(idx, idx)]) * np.linalg.det(R[np.ix_(idx, idx)]))
    gvif_adj = gvif ** (1.0 / (2.0 * d))
    
    gvif_rows.append({
        'Variable': var,
        'GVIF': round(gvif, 4),
        'df': d,
        'GVIF^(1/(2df))': round(gvif_adj, 4),
        'Equiv_VIF': round(gvif_adj**2, 4),
        'Status': '⚠ HIGH' if gvif_adj > 3.16 else '✓ OK'
    })

gvif_df = pd.DataFrame(gvif_rows)

print('\n' + '=' * 85)
print('GENERALIZED VIF (GVIF) — Fox & Monette (1992)')
print('GVIF^(1/(2·df)) > √10 ≈ 3.16 → problematic')
print('=' * 85)
print(gvif_df.to_string(index=False))
print('=' * 85)

high = gvif_df[gvif_df['GVIF^(1/(2df))'] > 3.16]
if len(high) > 0:
    print(f"⚠ High: {list(high['Variable'])}")
else:
    print("✓ No problematic multicollinearity")

gvif_df.to_excel('gvif_results_revised.xlsx', index=False)
print('Saved: gvif_results_revised.xlsx')


## PATCH 5: Update FEATURES and Re-encode X_encoded
📌 Replace cell 4.2 (feature matrix cell)

**Same variable names:** `FEATURES`, `X_encoded`  
Adds `province` and replaces `multiple_birth` dummies with single `is_multiple`


In [ ]:
# PATCH 5: Update FEATURES list + rebuild X_encoded
# REPLACES cell 4.2

# Updated FEATURES — same variable name as your original
FEATURES = ['age_group', 'marital_status', 'race_mother',
            'Gender ', 'Hospital or Not', 'province']

# Build X_encoded — same variable name as your original
X_encoded = pd.get_dummies(df[FEATURES], drop_first=True, dtype=float)

# Add binary is_multiple (replaces multi-level multiple_birth dummies)
X_encoded['is_multiple'] = df['is_multiple'].values

# Add continuous birth weight — same column name as your original
X_encoded['Birth_Weight(grams)'] = df['Birth_Weight(grams)'].values

# Same variable name
X_const = sm.add_constant(X_encoded)

print(f'Feature matrix: {X_encoded.shape[0]:,} rows x {X_encoded.shape[1]} features')
print('\nFeatures:')
for f in X_encoded.columns:
    print(f'  {f}')


## PATCH 6: Chi-Square Tests (Updated)
📌 Replace chi-square cell (cell 4.1)

Same variable: `chi2_df`


In [ ]:
# PATCH 6: Chi-square with collapsed parity + province
# Same variable name: chi2_df

PREDICTORS_CAT = ['age_group', 'marital_status', 'race_mother',
                  'Gender ', 'Hospital or Not', 'province']

chi2_rows = []
for var in PREDICTORS_CAT:
    ct = pd.crosstab(df['parity_collapsed'], df[var])
    chi2_val, p_val, dof, _ = stats.chi2_contingency(ct)
    chi2_rows.append({
        'Variable'   : var,
        'Chi2'       : round(chi2_val, 2),
        'df'         : dof,
        'p-value'    : round(p_val, 4),
        'Significant': 'Yes ***' if p_val < 0.001 else ('Yes **' if p_val < 0.01 else ('Yes *' if p_val < 0.05 else 'No'))
    })

chi2_df = pd.DataFrame(chi2_rows)
print('=' * 65)
print('CHI-SQUARE TESTS: PREDICTOR vs COLLAPSED PARITY (1/2/3/4+)')
print('=' * 65)
print(chi2_df.to_string(index=False))

chi2_df.to_excel('results_chisquare.xlsx', index=False)
print('\nSaved: results_chisquare.xlsx')


## PATCH 7: Ordinal Model with Collapsed Parity
📌 Replace ordinal model cells (5.1 + 5.2)

Same variables: `y_ord`, `ord_model`, `ord_result`


In [ ]:
# PATCH 7: Ordinal model — collapsed parity (1/2/3/4+)
# Same variable names: y_ord, ord_model, ord_result

y_ord = df['parity_collapsed'].astype(int).values

print('Outcome — collapsed parity:')
for val, cnt in zip(*np.unique(y_ord, return_counts=True)):
    labels = {1:'First', 2:'Second', 3:'Third', 4:'Fourth+'}
    print(f'  {labels.get(val, str(val)):8s} (parity {val}): {cnt:,}')

print('\nFitting Ordinal Logistic Regression (collapsed 4 categories)...')

ord_model = OrderedModel(y_ord, X_encoded, distr='logit')
ord_result = ord_model.fit(method='bfgs', maxiter=5000, disp=False)

print('Model fitted successfully.')
print(ord_result.summary())

print(f'\nLog-Likelihood : {ord_result.llf:.4f}')
print(f'AIC            : {ord_result.aic:.4f}')
print(f'BIC            : {ord_result.bic:.4f}')
print(f'Converged      : {ord_result.mle_retvals["converged"]}')


## PATCH 8: Count Model
📌 Replace count model cells (6.x)

Same variables: `y_count`, `poisson_m`, `nb_m`, `best_count_m`, `best_count_name`


In [ ]:
# PATCH 8: Count model — same variable names
# y_count, poisson_m, nb_m, best_count_m, best_count_name

y_count = df['parity'].astype(int).values

poisson_m = sm.GLM(y_count, X_const, family=sm.families.Poisson()).fit()
print('POISSON REGRESSION RESULTS')
print(poisson_m.summary())

# Negative Binomial
nb_m = sm.GLM(y_count, X_const, family=sm.families.NegativeBinomial()).fit()
print('\nNEGATIVE BINOMIAL RESULTS')
print(nb_m.summary())

# Select best
USE_NB = nb_m.aic < poisson_m.aic
best_count_m = nb_m if USE_NB else poisson_m
best_count_name = 'Negative Binomial' if USE_NB else 'Poisson'

print(f'\nPoisson AIC: {poisson_m.aic:.2f} | NB AIC: {nb_m.aic:.2f}')
print(f'Selected model: {best_count_name}')


## PATCH 9: Binary Logistic
📌 Replace binary logistic cell (7.1)

Same variables: `y_bin`, `logit_m`


In [ ]:
# PATCH 9: Binary logistic — same variable names
# y_bin, logit_m

y_bin = df['parity_binary'].values

logit_m = sm.Logit(y_bin, X_const).fit()

print('BINARY LOGISTIC REGRESSION RESULTS')
print(logit_m.summary())


## PATCH 10: Classification Disclaimer
📌 Insert before ROC/confusion matrix cell


In [ ]:
# PATCH 10: Disclaimer
print('=' * 70)
print('NOTE: Classification metrics below are SUPPLEMENTARY DIAGNOSTICS ONLY.')
print('PRIMARY OBJECTIVE: Identify predictor EFFECTS on parity,')
print('NOT to build a predictive model.')
print('=' * 70)


## PATCH 11: ŷ Hat Notation
📌 Replace/add after model equations


In [ ]:
# PATCH 11: Equations with hat notation

predictor_names = list(X_encoded.columns)
n_pred = len(predictor_names)

print('=' * 75)
print('FITTED MODEL EQUATIONS')
print('=' * 75)

# Model 1: Ordinal
print('\n── MODEL 1: ORDINAL LOGISTIC ──')
print('logit[ P̂(Y ≤ j | x) ] = α̂ⱼ + Σ β̂ₖxₖ')
params = ord_result.params
for i in range(n_pred):
    if ord_result.pvalues[i] < 0.05:
        s = '+' if params[i] >= 0 else '-'
        print(f'  {s} {abs(params[i]):.4f} × {predictor_names[i]}')
n_thresh = int(df['parity_collapsed'].nunique()) - 1
for j in range(n_thresh):
    idx = n_pred + j
    if idx < len(params):
        print(f'  α̂_{j+1}/{j+2} = {params[idx]:.4f}')

# Model 2: Count
print(f'\n── MODEL 2: {best_count_name.upper()} ──')
print('log(μ̂ᵢ) = β̂₀ + Σ β̂ₖxₖ')
print(f'  β̂₀ = {best_count_m.params.iloc[0]:.4f}')
for name in predictor_names:
    if name in best_count_m.params.index and best_count_m.pvalues[name] < 0.05:
        c = best_count_m.params[name]
        s = '+' if c >= 0 else '-'
        print(f'  {s} {abs(c):.4f} × {name}')

# Model 3: Binary
print('\n── MODEL 3: BINARY LOGISTIC ──')
print('logit(p̂) = β̂₀ + Σ β̂ₖxₖ')
print(f'  β̂₀ = {logit_m.params.iloc[0]:.4f}')
for name in predictor_names:
    if name in logit_m.params.index and logit_m.pvalues[name] < 0.05:
        c = logit_m.params[name]
        s = '+' if c >= 0 else '-'
        print(f'  {s} {abs(c):.4f} × {name}')
print(f'  p̂ = 1 / (1 + exp(−logit(p̂)))')
print('=' * 75)


## PATCH 12: Full OR/IRR Tables
📌 Replace OR table cells

Same variables: `ord_or`, `count_irr`, `binary_or`


In [ ]:
# PATCH 12: Full-precision OR/IRR tables
# Same variable names: ord_or, count_irr, binary_or

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.width', 120)

predictor_names = list(X_encoded.columns)
n_pred = len(predictor_names)

# Ordinal ORs — same var name
ord_or = pd.DataFrame({
    'OR': np.exp(ord_result.params[:n_pred]).round(4),
    'CI_lower': np.exp(ord_result.conf_int().iloc[:n_pred, 0]).round(4),
    'CI_upper': np.exp(ord_result.conf_int().iloc[:n_pred, 1]).round(4),
    'p_value': ord_result.pvalues[:n_pred].round(6),
    'Sig': [('***' if p<.001 else ('**' if p<.01 else ('*' if p<.05 else ''))) 
            for p in ord_result.pvalues[:n_pred]]
}, index=predictor_names)

print('MODEL 1 — CUMULATIVE ODDS RATIOS')
print(ord_or.to_string())
ord_or.to_excel('model1_ordinal_OR.xlsx')

# Count IRRs — same var name
count_irr = pd.DataFrame({
    'IRR': np.exp(best_count_m.params[1:]).round(4),
    'CI_lower': np.exp(best_count_m.conf_int().iloc[1:, 0]).round(4),
    'CI_upper': np.exp(best_count_m.conf_int().iloc[1:, 1]).round(4),
    'p_value': best_count_m.pvalues[1:].round(6),
    'Sig': [('***' if p<.001 else ('**' if p<.01 else ('*' if p<.05 else ''))) 
            for p in best_count_m.pvalues[1:]]
}, index=predictor_names)

print(f'\nMODEL 2 — {best_count_name.upper()} IRRs')
print(count_irr.to_string())
count_irr.to_excel('model2_count_IRR.xlsx')

# Binary ORs — same var name
binary_or = pd.DataFrame({
    'OR': np.exp(logit_m.params[1:]).round(4),
    'CI_lower': np.exp(logit_m.conf_int().iloc[1:, 0]).round(4),
    'CI_upper': np.exp(logit_m.conf_int().iloc[1:, 1]).round(4),
    'p_value': logit_m.pvalues[1:].round(6),
    'Sig': [('***' if p<.001 else ('**' if p<.01 else ('*' if p<.05 else ''))) 
            for p in logit_m.pvalues[1:]]
}, index=predictor_names)

print(f'\nMODEL 3 — BINARY ODDS RATIOS')
print(binary_or.to_string())
binary_or.to_excel('model3_binary_OR.xlsx')

print('\n✓ All saved')


## ✅ Quick Reference

| Patch | Replaces | Variables kept same |
|-------|----------|-------------------|
| 1 | New cell after 2.9 | adds `parity_collapsed` |
| 2 | New cell | adds `is_multiple` |
| 3 | New cell | adds `province` |
| 4 | Cell 4.3 (VIF) | `gvif_df` (new) |
| 5 | Cell 4.2 (features) | `FEATURES`, `X_encoded`, `X_const` |
| 6 | Cell 4.1 (chi-square) | `chi2_df`, `PREDICTORS_CAT` |
| 7 | Cell 5.1+5.2 (ordinal) | `y_ord`, `ord_model`, `ord_result` |
| 8 | Cell 6.x (count) | `y_count`, `poisson_m`, `nb_m`, `best_count_m`, `best_count_name` |
| 9 | Cell 7.1 (binary) | `y_bin`, `logit_m` |
| 10 | New cell before ROC | — |
| 11 | Equation cells | — |
| 12 | OR table cells | `ord_or`, `count_irr`, `binary_or` |

All downstream cells (forest plots, ROC, confusion matrix, Word doc export) will work unchanged because the variable names are identical.
